# Computer Vision Evaluation

## Overview

Evaluating CV models correctly requires task-specific metrics, error analysis, and robustness checks. A single aggregate metric rarely tells the full story.

**Metrics by task:**

| Task | Primary metric | Secondary |
|---|---|---|
| Classification | Top-1 Accuracy | Per-class F1, AUC |
| Detection | mAP@0.5 | mAP@[.5:.95], AR |
| Segmentation | mIoU | mDice, per-class IoU |
| Retrieval | mAP | Recall@K |

**Beyond accuracy:**
- Confusion matrix: which classes are confused
- Failure mode analysis: systematic patterns in errors
- Calibration: do confidence scores reflect true accuracy
- Robustness: performance under brightness, blur, rotation shifts
- Class activation maps: what the model attends to

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from copy import deepcopy

torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
N_CLASSES = 4
CLASS_NAMES = ['Excellent','Good','Moderate','Poor']

def make_patch(label, size=64, seed=0):
    r = np.random.default_rng(seed)
    # Difficulty increases from label 0->3 (more similar colours)
    green = max(60, 150 - label*30); red = 50 + label*25
    img = np.stack([r.integers(red-20, red+20, (size,size)),
                    r.integers(green-20, green+20, (size,size)),
                    r.integers(30, 70, (size,size))], axis=2)
    return Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))

class PatchDS(Dataset):
    def __init__(self, imgs, labels, tfm):
        self.imgs, self.labels, self.tfm = imgs, labels, tfm
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return self.tfm(self.imgs[i]), torch.tensor(self.labels[i], dtype=torch.long)

all_imgs   = [make_patch(lbl, seed=i+lbl*300) for lbl in range(N_CLASSES) for i in range(100)]
all_labels = [lbl for lbl in range(N_CLASSES) for _ in range(100)]
eval_tfm   = T.Compose([T.Resize((64,64)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
train_tfm  = T.Compose([T.Resize((64,64)), T.RandomHorizontalFlip(),
    T.ColorJitter(0.3,0.3,0.2), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
sp1, sp2 = int(0.7*len(all_imgs)), int(0.85*len(all_imgs))
train_dl = DataLoader(PatchDS(all_imgs[:sp1], all_labels[:sp1], train_tfm), 32, shuffle=True)
val_dl   = DataLoader(PatchDS(all_imgs[sp1:sp2], all_labels[sp1:sp2], eval_tfm), 32)
test_dl  = DataLoader(PatchDS(all_imgs[sp2:], all_labels[sp2:], eval_tfm), 32)
print(f"Train={sp1}, Val={sp2-sp1}, Test={len(all_imgs)-sp2}")

---
## Training a Baseline Classifier

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(2),
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(128*4, 64), nn.ReLU(), nn.Linear(64, n))
    def forward(self, x): return self.net(x)

model = SimpleCNN().to(device)
opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
crit  = nn.CrossEntropyLoss()
best_acc, best_state = 0, None
for ep in range(50):
    model.train()
    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        acc = np.mean([(model(Xb.to(device)).argmax(1)==yb.to(device)).float().mean().item()
                       for Xb, yb in val_dl])
    if acc > best_acc:
        best_acc = acc; best_state = deepcopy(model.state_dict())
model.load_state_dict(best_state)
print(f"Val accuracy: {best_acc:.3f}")

---
## Confusion Matrix and Per-Class Report

In [ ]:
model.eval()
all_preds, all_true, all_probs = [], [], []
with torch.no_grad():
    for Xb, yb in test_dl:
        out   = model(Xb.to(device))
        probs = torch.softmax(out, dim=1).cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds); all_true.extend(yb.numpy())
        all_probs.append(probs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_probs = np.vstack(all_probs)

print(classification_report(all_true, all_preds, target_names=CLASS_NAMES))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(confusion_matrix(all_true, all_preds),
    display_labels=CLASS_NAMES).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')
# Normalised confusion matrix
cm_norm = confusion_matrix(all_true, all_preds, normalize='true')
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], colorbar=False, cmap='Blues', values_format='.2f')
axes[1].set_title('Normalised (row = true class)')
plt.tight_layout(); plt.show()

---
## Robustness Under Distribution Shift

In [ ]:
def apply_corruption(imgs, corruption_type, severity=0.3):
    corrupted = []
    for img in imgs:
        arr = np.array(img).astype(float)
        if corruption_type == 'brightness':
            arr = arr * (1 + severity)
        elif corruption_type == 'noise':
            arr = arr + np.random.normal(0, severity*60, arr.shape)
        elif corruption_type == 'blur':
            from scipy.ndimage import gaussian_filter
            sigma = severity * 5
            arr = np.stack([gaussian_filter(arr[:,:,c], sigma) for c in range(3)], axis=2)
        corrupted.append(Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)))
    return corrupted

test_imgs_raw = all_imgs[sp2:]
test_lbls_raw = all_labels[sp2:]

def eval_corrupted(imgs, labels, corruption=None, severity=0.3):
    if corruption:
        imgs = apply_corruption(imgs, corruption, severity)
    ds = PatchDS(imgs, labels, eval_tfm)
    dl = DataLoader(ds, batch_size=32)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in dl:
            out = model(Xb.to(device))
            correct += (out.argmax(1)==yb.to(device)).sum().item()
            total += len(yb)
    return correct / total

results_rob = {}
results_rob['Clean']      = eval_corrupted(test_imgs_raw, test_lbls_raw)
results_rob['Brightness'] = eval_corrupted(test_imgs_raw, test_lbls_raw, 'brightness')
results_rob['Noise']      = eval_corrupted(test_imgs_raw, test_lbls_raw, 'noise')
results_rob['Blur']       = eval_corrupted(test_imgs_raw, test_lbls_raw, 'blur')
fig, ax = plt.subplots(figsize=(8,4))
names, accs = list(results_rob.keys()), list(results_rob.values())
colors = ['steelblue' if n=='Clean' else '#e74c3c' for n in names]
ax.bar(names, accs, color=colors, edgecolor='white')
ax.axhline(results_rob['Clean'], color='grey', lw=1, linestyle='--')
ax.set_ylabel('Accuracy'); ax.set_ylim([0,1])
ax.set_title('Robustness Under Distribution Shifts')
plt.tight_layout(); plt.show()
for name, acc in results_rob.items():
    drop = results_rob['Clean'] - acc
    print(f"  {name:12s}: {acc:.3f} (drop: {drop:+.3f})")

In [ ]:
# Top-K accuracy and per-class AUC
def topk_accuracy(probs, labels, k=2):
    topk = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([labels[i] in topk[i] for i in range(len(labels))])

top1 = (all_preds == all_true).mean()
top2 = topk_accuracy(all_probs, all_true, k=2)
print(f"Top-1 accuracy: {top1:.3f}")
print(f"Top-2 accuracy: {top2:.3f}")
print("  Top-2 useful for ordinal classes where adjacent errors are acceptable")

# Per-class one-vs-rest AUC
print("\nPer-class AUC (one-vs-rest):")
for cls in range(N_CLASSES):
    binary_true = (all_true == cls).astype(int)
    auc = roc_auc_score(binary_true, all_probs[:, cls])
    print(f"  {CLASS_NAMES[cls]:12s}: AUC={auc:.3f}")

# Error analysis: show most confidently wrong predictions
errors_idx = np.where(all_preds != all_true)[0]
if len(errors_idx):
    confidences = all_probs[errors_idx, all_preds[errors_idx]]
    worst_idx   = errors_idx[np.argsort(confidences)[-5:]][::-1]
    print("\nMost confidently wrong predictions:")
    for idx in worst_idx:
        conf = all_probs[idx, all_preds[idx]]
        print(f"  True: {CLASS_NAMES[all_true[idx]]}, Pred: {CLASS_NAMES[all_preds[idx]]}, Confidence: {conf:.3f}")

---

## Common Pitfalls

**1. Not examining which specific images the model gets wrong**  
Aggregate metrics hide systematic failure modes. A model achieving 85% accuracy may be perfectly accurate on well-lit images but fail entirely on images with poor lighting or unusual viewpoints. Always inspect a random sample of misclassified images to identify systematic patterns — these usually point to training data gaps.

**2. Using top-1 accuracy for fine-grained classification with visually similar classes**  
For tasks with many similar classes (e.g. 10 shades of vegetation health), a model that confuses adjacent classes (Excellent vs Good) makes less harmful errors than one that confuses opposite classes (Excellent vs Poor). Report top-2 or top-3 accuracy alongside top-1 for ordinal or hierarchical class structures.

**3. Not testing robustness under realistic distribution shifts before deployment**  
Models evaluated only on clean test images often degrade severely under practical conditions: different weather, sensor calibration drift, seasonal lighting changes. Always test with brightness, contrast, noise, and blur corruptions before deployment, and monitor performance after deployment.

**4. Reporting only overall accuracy without per-class breakdown for imbalanced datasets**  
With imbalanced classes, overall accuracy is dominated by the majority class. Always report per-class precision, recall, and F1, and inspect the normalised confusion matrix (rows sum to 1) to understand systematic confusions between specific class pairs.

**5. Not distinguishing between top-1 accuracy on the validation set and the test set**  
Accuracy reported on the validation set is inflated because the model was selected based on its validation performance. The unbiased estimate of generalisation is accuracy on a held-out test set that was never used for any decision during training or model selection. Always clearly label which split produced each reported metric.
---
*python_methods_library - Samantha McGarrigle*